# cbj · 전체 데이터 EDA + XGB 베이스라인
**Kaggle 전체 데이터 실행.** 로컬에서 확인한 20행 미리보기와 별개로, 이 버전은 Kaggle에서 원본 CSV를 전체 읽습니다. A~E의 집계 모집단은 전체 train입니다. test와 제출 양식은 구조 점검에만 사용합니다.

메틸레이션 기능 지도는 유전자를 탐색하는 관점입니다. 모든 입력 유전자와 원본 변이 표기는 유지합니다. 경로 34개 유전자의 초안은 전체 경로 목록이 아닙니다. 모델은 전체 유전자 입력을 사용하며 경로 요약으로 대체하지 않습니다.

**실행 순서:** 전체 EDA → 고정 80/20 내부 검증 → 같은 설정으로 전체 train 재학습. 모델 선택이나 튜닝 없이 XGB 100회 설정을 사용합니다. 정답이 없는 test의 성능은 보고하지 않으며 제출하지 않습니다.

In [ ]:
from pathlib import Path
import os, re, json, itertools, sys, platform
os.environ.setdefault('MPLCONFIGDIR', '/tmp/cbj_mpl')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager, colors
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from IPython.display import display, Markdown, Image, HTML

# Full source reads are authorized ONLY in Kaggle for this notebook.
assert Path('/kaggle/input').exists(), 'Full-data execution is restricted to Kaggle; do not run locally.'
matches=list(Path('/kaggle/input').rglob('cbj_gene_pathway.csv'))
assert len(matches)==1
DATA=matches[0].parent; MAP=matches[0]
OUT=Path('/kaggle/working/cbj_full_results'); OUT.mkdir(exist_ok=True,parents=True)
frames={name:pd.read_csv(DATA/name,dtype=str,keep_default_na=False)
        for name in ['train.csv','test.csv','sample_submission.csv']}
train,test,submission=[frames[k] for k in frames]
N=len(train)
print({name:{'rows':len(df),'columns':len(df.columns)} for name,df in frames.items()},flush=True)
features=[c for c in train.columns if c not in ['ID','SUBCLASS']]
assert N>20 and train.SUBCLASS.ne('').all()
assert features==[c for c in test.columns if c!='ID']
assert list(submission.columns)==['ID','SUBCLASS']
assert train.ID.is_unique and test.ID.is_unique
mapping=pd.read_csv(MAP,keep_default_na=False)
for c in ['present_in_train','present_in_test']:
    mapping[c]=mapping[c].astype(str).str.lower().eq('true')
assert mapping.present_in_train.tolist()==mapping.gene.isin(features).tolist()
assert not mapping.duplicated(['gene','stage']).any()
font_candidates=[DATA/'NanumGothic-Regular.ttf',Path('.cbj_kaggle/data/NanumGothic-Regular.ttf')]
font=next((p for p in font_candidates if p.exists()),None)
if font:
    font_manager.fontManager.addfont(str(font)); plt.rcParams['font.family']=font_manager.FontProperties(fname=font).get_name()
plt.rcParams.update({'font.size':11,'axes.unicode_minus':False,'figure.dpi':120,'savefig.dpi':160,
                     'axes.spines.top':False,'axes.spines.right':False})
SCOPE=f'전체 train 기반 · n={N:,}명 · test 정답 분석 제외'
stages=['Folate','Met-SAM','DNA','Histone','Met-axis']
names={'Folate':'일탄소·엽산','Met-SAM':'메티오닌·SAM/SAH','DNA':'DNA 메틸화/탈메틸화','Histone':'히스톤·염색질','Met-axis':'대사물질 조절 축'}
stage_genes={s:mapping.loc[(mapping.stage==s)&mapping.present_in_train,'gene'].drop_duplicates().tolist() for s in stages}
stage_total={s:mapping.loc[mapping.stage==s,'gene'].nunique() for s in stages}
# Preserve raw strings. Empty and explicit missing markers are unknown, never WT.
missing_markers={'','NA','N/A','NAN','NULL','NONE','<NA>','.'}
raw=train[features]
clean=raw.apply(lambda col:col.str.strip())
missing=clean.apply(lambda col:col.str.upper().isin(missing_markers))
mut=clean.ne('WT')&~missing
def stage_state(genes):
    if not genes: return pd.Series(np.nan,index=train.index)
    positive=mut[genes].any(axis=1)
    unknown=missing[genes].any(axis=1)&~positive
    return positive.astype(float).mask(unknown)
state=pd.DataFrame({s:stage_state(stage_genes[s]) for s in stages})
burden=mut.sum(axis=1)
classes=sorted(train.SUBCLASS.unique())
counts=train.SUBCLASS.value_counts().reindex(classes)
coverage=pd.DataFrame([{'단계':names[s],'열 존재':len(stage_genes[s]),'초안 유전자':stage_total[s],
                       '열 없는 유전자':', '.join(mapping.loc[(mapping.stage==s)&~mapping.present_in_train,'gene'])} for s in stages])
def save(fig,name):
    fig.savefig(OUT/(name+'.png'),bbox_inches='tight',facecolor='white')
    plt.close(fig); display(Image(filename=str(OUT/(name+'.png'))))
def heat(ax,data,xlabels,ylabels,annot,vmin=0,vmax=1,cmap='Blues'):
    cm=plt.get_cmap(cmap).copy();cm.set_bad('#d8d8d8')
    im=ax.imshow(np.ma.masked_invalid(np.array(data,dtype=float)),vmin=vmin,vmax=vmax,cmap=cm,aspect='auto')
    ax.set_xticks(range(len(xlabels)),xlabels,rotation=0,fontsize=10)
    ax.set_yticks(range(len(ylabels)),ylabels)
    for i in range(len(ylabels)):
        for j in range(len(xlabels)):
            v=np.array(data,dtype=float)[i,j]
            ax.text(j,i,annot[i][j],ha='center',va='center',fontsize=10,
                    color='white' if np.isfinite(v) and v>vmax*.6 else '#20252b')
    return im

## A·B | 전체 SUBCLASS 구성과 기능 단계
**A:** 전체 train의 클래스별 환자 수. **B:** 데이터에 존재하는 해당 단계 유전자 중 하나 이상에 비결측·비WT 표기가 있는 환자의 비율(동의변이 포함).

셀은 `양성 환자/판정 가능한 환자`, `?`는 미확인 수입니다. 변이가 없고 결측이 있으면 미확인입니다. 데이터에 열이 없는 유전자는 WT가 아니며, 이 비율은 관측 가능한 부분 경로에만 해당합니다. 유전자 수·포함 범위가 달라 단계 중요도 순위로 해석하지 않습니다.

In [ ]:
b_rows=[]; ratios=[]; annotations=[]
for cl in classes:
    ix=train.SUBCLASS.eq(cl); rr=[]; aa=[]
    for s in stages:
        z=state.loc[ix,s]; num=int(z.eq(1).sum()); den=int(z.notna().sum()); unk=int(z.isna().sum())
        rr.append(num/den if den else np.nan); aa.append(f'{num}/{den}'+(f'\n?{unk}' if unk else ''))
        b_rows.append(dict(SUBCLASS=cl,stage=s,positive=num,known=den,unknown=unk,class_n=int(ix.sum()),genes_present=len(stage_genes[s])))
    ratios.append(rr); annotations.append(aa)
fig,(ax,bx)=plt.subplots(1,2,figsize=(17,max(8,len(classes)*.36)),gridspec_kw={'width_ratios':[1,2.6]})
fig.suptitle('A·B  SUBCLASS 구성과 기능 단계 변이 표기',fontsize=18,y=1.02)
ax.barh(classes,counts,color='#467eae');ax.invert_yaxis();ax.set_xlim(0,counts.max()*1.24)
ax.set_xlabel(f'환자 수 (명) / 전체 train {N:,}명')
ax.set_title('A  전체 train 클래스 구성',loc='left',pad=14)
for i,n in enumerate(counts):ax.text(n+counts.max()*.015,i,f'{n}명',va='center')
labels=[names[s].replace('·','·\n',1)+f'\n열 {len(stage_genes[s])}/{stage_total[s]}개' for s in stages]
im=heat(bx,ratios,labels,[f'{c} (n={counts[c]})' for c in classes],annotations)
bx.set_title('B  변이 표기 환자 / 판정 가능한 환자',loc='left',pad=14)
fig.colorbar(im,ax=bx,fraction=.025,pad=.02,label='환자 비율',ticks=[0,.5,1])
fig.text(.02,-.035,SCOPE+'\nB: 셀 = 분자/분모; ? = 미확인. 경로별 유전자 수·관측 범위 차이로 중요도를 비교할 수 없음.',fontsize=11)
fig.tight_layout();save(fig,'A_B_subclass_pathway')
display(Markdown(f'**현재 보이는 것:** {len(classes)}개 클래스, 클래스당 {counts.min()}~{counts.max()}명입니다. DNA 단계는 {len(stage_genes["DNA"])}개 열만 관측되어 0을 전체 DNA 메틸화 경로의 무변이로 읽을 수 없습니다. **다음 확인:** 전체 변이 유전자 수와 공유 유전자를 고려해도 단계별 차이가 남는지.'))

## 분석 지도 | 측정 결과와 분리한 생물학적 개념도
화살표는 문헌의 생물학적 관계입니다. 이 데이터는 대사 흐름·SAM/SAH 농도·메틸화량을 측정하지 않습니다. 유전자 기능 소속은 근거를 확인했지만, 관측된 특정 변이가 기능을 바꾼다는 판정은 하지 않습니다. 초안은 완전한 경로 목록이 아닙니다.

[엽산 대사](https://reactome.org/content/detail/R-HSA-196757), [SAM 회로](https://reactome.org/content/detail/R-HSA-156581), [DNA/히스톤 조절](https://reactome.org/content/detail/R-HSA-212165), [IDH 변이 원논문](https://www.nature.com/articles/nature10860)을 출발점으로 사용했습니다. 유전자별 추가 근거는 아래 매핑표에 있습니다.

In [ ]:
fig,ax=plt.subplots(figsize=(15,8));ax.set_xlim(0,16);ax.set_ylim(0,7.5);ax.axis('off')
fig.suptitle('메틸레이션 기능 지도 | 생물학적 개념도',fontsize=18)
def box(x,y,w,h,text):
    ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.15',facecolor='#f1f5f8',edgecolor='#54718b'))
    ax.text(x+w/2,y+h/2,text,ha='center',va='center',fontsize=11)
def arrow(a,b,text='',rad=0,style='->'):
    ax.add_patch(FancyArrowPatch(a,b,arrowstyle=style,mutation_scale=15,connectionstyle=f'arc3,rad={rad}',color='#54718b'))
    if text: ax.text((a[0]+b[0])/2,(a[1]+b[1])/2+.14,text,ha='center',fontsize=10)
box(.2,4.4,3.4,2,'일탄소·엽산\nSHMT1/2, MTHFD1/2, DHFR\nMTHFR → 5-methyl-THF\nMTHFD1L [열 없음]')
box(5.1,5.7,3.1,.75,'메티오닌 → SAM\nMAT1A/2A/2B [열 없음]')
box(5.1,3.5,3.1,.75,'SAH ⇄ 호모시스테인\nAHCY [열 있음]')
arrow((6.1,4.35),(6.1,5.55),'MTR [없음] / MTRR',rad=.5)
arrow((8.3,5.9),(8.3,4.0),'',rad=-.5)
ax.text(7.1,4.8,'메틸기 전달',fontsize=10)
arrow((3.75,5.4),(5.0,5.9),'재메틸화')
arrow((5.0,4.0),(3.2,4.2),'엽산 재생',rad=-.2)
box(10.3,5.4,4.2,1.5,'DNA 메틸화·탈메틸화\nDNMT1 [있음] / DNMT3A/B [없음]\nTET1/2/3, TDG [열 없음]')
box(10.3,2.9,4.2,1.5,'히스톤 메틸화·염색질\nEZH2, EED, KMT2D, KDM6B, ATRX\nSUZ12 [열 없음]')
arrow((8.3,6.3),(10.15,6.3),'SAM 공급')
arrow((8.3,5.7),(10.15,4.15),'SAM 공급')
arrow((12.4,4.5),(12.4,5.2),'',style='<->')
ax.text(12.7,4.8,'조절 연결',fontsize=10)
box(.3,.4,7.8,1.7,'대사물질 조절 축: IDH1/2, FH, SDHA/B/C/D\nalpha-KG 의존 TET·KDM과 연결\n일부 기능 변화 → 2-HG / 푸마르산 / 숙신산 축적 가능\n특정 변이의 효과와 농도는 이번 데이터에서 미측정')
arrow((8.25,1.5),(11,2.7),'탈메틸화 조절',rad=.1)
ax.plot([8.25,15.2,15.2],[1.1,1.1,6.2],color='#54718b',lw=1)
arrow((15.2,6.2),(14.7,6.2))
ax.text(4.9,2.65,'호모시스테인 → 황전이 분기 등\n(이번 초안 정량 범위 밖)',fontsize=10)
arrow((6,3.35),(6,3.0))
fig.text(.04,.015,'전체 데이터 EDA의 분석 지도 · 수치 표본/단위: 해당 없음 · 화살표는 측정한 흐름·인과관계가 아님',fontsize=11)
save(fig,'concept_map')
display(coverage)
display(mapping.rename(columns={'gene':'유전자','stage':'기능군','role':'역할','evidence':'근거','mapping_status':'매핑 상태','multi_membership':'중복 소속'}))

## C | 전체 데이터의 단계 → 유전자 → 변이 표기
표기 조각을 세며 독립 생물학적 변이 수로 해석하지 않습니다. 복수 표기의 모든 조각과 원본 문자열을 CSV에 보존합니다. 단계 중복 소속은 합산 시 중복될 수 있습니다. 단백질 위치와 아미노산 전후는 단순 문법에서만 파싱하며 불확실성은 별도 열에 남깁니다.

화면의 원본 예시는 일부만 표시하지만 **유형 집계와 내려받는 표는 전체 train 기준**입니다. 큰 원본 표는 압축 CSV로 저장합니다.

In [ ]:
types=['동의변이','아미노산 치환','종결','프레임시프트','결실','삽입','복합·미분류']
aa=set('ACDEFGHIKLMNPQRSTVWY')
def parse_token(token):
    d=dict(type='복합·미분류',aa_from='',position='',aa_to='',parse_status='불확실')
    t=token.removeprefix('p.')
    m=re.fullmatch(r'([A-Z])(\d+)([A-Z*]|=)',t)
    if m and m[1] in aa and (m[3] in aa or m[3] in '*='):
        a,p,b=m.groups();d.update(aa_from=a,position=int(p),aa_to=b,parse_status='표기 문법 해석')
        d['type']='종결' if b=='*' else ('동의변이' if b==a or b=='=' else '아미노산 치환')
    elif re.fullmatch(r'[A-Z]+\d+(?:[A-Z])?fs(?:\*\d+|Ter\d+)?',t):
        d['type']='프레임시프트';m=re.match(r'([A-Z]+)(\d+)',t)
        if len(m[1])==1 and m[1] in aa:d.update(aa_from=m[1],position=int(m[2]),parse_status='유형·시작 위치만 해석')
        else:d['parse_status']='유형 추정; 다문자 위치 의미 불확실'
    elif 'delins' in t:d['parse_status']='복합 delins 표기; 좌표 미해석'
    elif re.fullmatch(r'[A-Z]?\d+(?:_[A-Z]?\d+)?del',t):d.update(type='결실',parse_status='유형만 해석')
    elif re.fullmatch(r'[A-Z]?\d+(?:_[A-Z]?\d+)?ins[A-Z]+',t):d.update(type='삽입',parse_status='유형만 해석')
    return d
records=[]
# Traverse only observed non-WT, nonmissing cells, retaining every original token.
for gene in features:
    for i,value in train.loc[mut[gene],gene].items():
        tokens=[t for t in re.split(r'[\s;,|]+',value.strip()) if t]
        for k,token in enumerate(tokens):
            records.append(dict(ID=train.loc[i,'ID'],SUBCLASS=train.loc[i,'SUBCLASS'],gene=gene,raw=value,token=token,
                                token_index=k+1,tokens_in_cell=len(tokens),**parse_token(token)))
annotations_all=pd.DataFrame(records)
mapped=annotations_all.merge(mapping[['gene','stage']],on='gene',how='inner')
ct=pd.crosstab(mapped.stage,mapped.type).reindex(index=stages,columns=types,fill_value=0).fillna(0).astype(int)
genes_seen=mapped.gene.drop_duplicates().tolist()
gt=pd.crosstab(mapped.drop_duplicates(['ID','gene','token_index']).gene,mapped.drop_duplicates(['ID','gene','token_index']).type).reindex(index=genes_seen,columns=types,fill_value=0).fillna(0).astype(int)
fig,(ax,bx)=plt.subplots(1,2,figsize=(17,max(6,len(genes_seen)*.32)),gridspec_kw={'width_ratios':[1.35,1]})
fig.suptitle('C  단계·유전자별 표기 유형',fontsize=18,y=1.02)
mx=max(int(ct.to_numpy().max()),1)
im=heat(ax,ct,types,[names[s] for s in stages],ct.astype(str).values.tolist(),vmax=mx)
ax.tick_params(axis='x',labelrotation=35);ax.set_title('단계별 표기 조각 수 (중복 소속 포함)',pad=14)
heat(bx,gt,types,genes_seen,gt.astype(str).values.tolist(),vmax=mx)
bx.tick_params(axis='x',labelrotation=35);bx.set_title('관측 유전자별 표기 조각 수',pad=14)
fig.text(.02,-.05,SCOPE+'\n0 = 전체 train에서 관측된 해당 유형 없음. 표기 수 ≠ 독립 생물학적 변이 수. 다른 유전자도 전체 annotations_all에 보존.',fontsize=11)
fig.tight_layout();save(fig,'C_variant_types')
cols=['stage','SUBCLASS','ID','gene','raw','token','type','aa_from','position','aa_to','parse_status']
display(mapped[cols].sort_values(['stage','gene','ID']).head(60))
display(Markdown('**현재 보이는 것:** 단계 수준에서 섞인 변이 유형을 유전자와 원본 표기로 구분할 수 있습니다. **다음 확인:** 동의변이 제외 및 위치별 반복 출현에서도 클래스 차이가 유지되는지.'))
display(Markdown('**전체 유전자 관점의 복수 표기 예시** (첫 8개 환자-유전자 셀 예시; 모든 조각 유지)'))
multi_keys=annotations_all.loc[annotations_all.tokens_in_cell.gt(1),['ID','gene']].drop_duplicates().head(8)
display(annotations_all.merge(multi_keys,on=['ID','gene'])[['ID','SUBCLASS','gene','raw','token','type','aa_from','position','aa_to','parse_status']])
display(Markdown('**파싱이 불확실한 실제 표기** (최대 12조각; 전체는 annotations_all에 보존)'))
display(annotations_all.loc[annotations_all.parse_status.str.contains('불확실'),['ID','gene','raw','token','type','parse_status']].head(12))
annotations_all.to_csv(OUT/'variant_annotations_full.csv.gz',index=False,encoding='utf-8-sig',compression='gzip')
mapped.to_csv(OUT/'mapped_variant_annotations_full.csv',index=False,encoding='utf-8-sig')
ct.to_csv(OUT/'C_stage_type_counts.csv',encoding='utf-8-sig')
gt.to_csv(OUT/'C_gene_type_counts.csv',encoding='utf-8-sig')
print('Full non-WT cells:',int(mut.sum().sum()),'notation fragments:',len(annotations_all),flush=True)

## D | 전체 train의 동시 출현과 SUBCLASS별 비교
셀은 동시 양성 환자 수/판정 가능한 환자 수입니다. `*`는 공유 유전자가 있는 쌍이며 오른쪽은 쌍별 공유 유전자를 제외한 민감도 확인입니다. 동시 출현을 상호작용이나 인과로 읽지 않습니다.

클래스별 그림은 환자 20명 이상인 클래스에서 기술적 비교를 제공합니다. 이 임계치는 통계적 검정력을 보장하지 않으며, 셀별 실제 분모를 함께 봅니다.

In [ ]:
co=np.zeros((5,5)); independent=np.zeros((5,5)); ann=[]; ann2=[]; overlap=[]
for i,a in enumerate(stages):
    ar=[];br=[]
    for j,b in enumerate(stages):
        shared=set(stage_genes[a])&set(stage_genes[b]) if a!=b else set()
        valid=state[a].notna()&state[b].notna()
        num=int((state[a].eq(1)&state[b].eq(1)).sum()); co[i,j]=num
        ar.append(f'{num}/{int(valid.sum())}'+('*' if shared else ''))
        sa=stage_state([g for g in stage_genes[a] if g not in shared]); sb=stage_state([g for g in stage_genes[b] if g not in shared])
        den=int((sa.notna()&sb.notna()).sum()); n=int((sa.eq(1)&sb.eq(1)).sum())
        independent[i,j]=n if den else np.nan;br.append(f'{n}/{den}' if den else '?')
        if i<j and shared:overlap.append(dict(stage_a=a,stage_b=b,shared_genes=','.join(sorted(shared)),observed_co=num,without_shared=n))
    ann.append(ar);ann2.append(br)
fig,(ax,bx)=plt.subplots(1,2,figsize=(15,6))
fig.suptitle('D  단계 간 동시 출현 | 환자 건수',fontsize=18,y=1.02)
lim=max(float(co.max()),1)
im=heat(ax,co,stages,stages,ann,vmax=lim);ax.set_title('공유 유전자 포함: 원래 단계 정의',pad=15)
heat(bx,independent,stages,stages,ann2,vmax=lim);bx.set_title('각 쌍의 공유 유전자 제외',pad=15)
fig.colorbar(im,ax=[ax,bx],fraction=.02,pad=.025,label='동시 양성 환자 수 (명)')
fig.text(.04,-.01,SCOPE+'\n셀=동시 양성/판정 가능 환자; * 공유 유전자 있음. 상호작용·인과관계를 의미하지 않음.',fontsize=11)
save(fig,'D_cooccurrence');display(pd.DataFrame(overlap))
display(Markdown('**현재 보이는 것:** 위 표에서 공유 유전자 제거 전후의 실제 동시 출현을 확인합니다. **다음 확인:** 변이 유전자 수를 맞추고도 두 단계의 별도 유전자에서 동시 출현이 남는지.'))
eligible=[cl for cl in classes if counts[cl]>=20]
cols_n=4;rows_n=int(np.ceil(len(eligible)/cols_n))
fig,axs=plt.subplots(max(rows_n,1),cols_n,figsize=(19,max(rows_n,1)*4.1),squeeze=False)
co_class=[]
for ax,cl in zip(axs.flat,eligible):
    z=state.loc[train.SUBCLASS.eq(cl)]; values=np.zeros((5,5)); texts=[]
    for i,a in enumerate(stages):
        row=[]
        for j,b in enumerate(stages):
            num=int((z[a].eq(1)&z[b].eq(1)).sum());den=int((z[a].notna()&z[b].notna()).sum())
            values[i,j]=num/den if den else np.nan
            shared=set(stage_genes[a])&set(stage_genes[b]) if a!=b else set()
            row.append(f'{num}/{den}'+('*' if shared else ''))
            co_class.append(dict(SUBCLASS=cl,stage_a=a,stage_b=b,positive=num,known=den,shared=','.join(sorted(shared))))
        texts.append(row)
    heat(ax,values,stages,stages,texts);ax.tick_params(axis='x',labelrotation=50)
    for t in ax.texts:t.set_fontsize(7)
    ax.set_title(f'{cl} (n={len(z)})')
for ax in list(axs.flat)[len(eligible):]:ax.axis('off')
fig.suptitle('D2  SUBCLASS별 단계 동시 출현 | 색상=환자 비율 0~1',fontsize=17,y=1.005)
fig.text(.02,-.012,SCOPE+'; 셀=동시 양성/판정 가능, * 공유 유전자. 표본 20명 미만 클래스는 집계 불충분.',fontsize=11)
fig.tight_layout();save(fig,'D2_subclass_cooccurrence')
pd.DataFrame(co_class).to_csv(OUT/'D_subclass_cooccurrence.csv',index=False)

## E | 전체 환자의 변이 유전자 수와 단계 배치
모든 입력 유전자의 비결측·비WT 셀 수로 **전체 환자**를 정렬합니다. 이는 독립 변이 개수나 TMB가 아닙니다. 개요 그림에는 모든 환자가 포함되지만 화면 해상도로 행들이 압축됩니다. 환자 ID·SUBCLASS·변이 유전자 수·단계 상태는 전체 CSV에서 정확히 확인할 수 있습니다.

옆의 SUBCLASS 패널은 각 환자의 클래스 위치를 점으로 표시합니다. 아래 상세 비교는 전체 자료에서 변이 유전자 수가 정확히 같고 단계가 다른 비교쌍 중 결정적으로 선택한 최대 8쌍입니다. 전체 분석을 이 예시로 제한하지 않습니다.

In [ ]:
order=burden.sort_values(kind='stable').index;patterns=state.loc[order]
patient_table=train[['ID','SUBCLASS']].assign(mutated_gene_cells=burden,missing_gene_cells=missing.sum(axis=1)).join(state)
patient_table.loc[order].to_csv(OUT/'E_all_patient_patterns.csv',index=False,encoding='utf-8-sig')
fig,(ax,bx,cx)=plt.subplots(1,3,figsize=(18,11),gridspec_kw={'width_ratios':[2.4,2.8,1.4]})
cm=plt.get_cmap('Blues').copy();cm.set_bad('#aaa')
ax.imshow(np.ma.masked_invalid(patterns.to_numpy()),vmin=0,vmax=1,cmap=cm,aspect='auto',interpolation='nearest',extent=[-.5,4.5,N-.5,-.5])
ax.set_xticks(range(5),stages);ax.set_ylabel('변이 유전자 수로 정렬한 환자 순번 (0부터)')
ax.set_title('전체 환자 × 단계: 파랑=양성, 흰색=WT, 회색=?',fontsize=11)
cl_idx={cl:i for i,cl in enumerate(classes)}
bx.scatter(train.loc[order,'SUBCLASS'].map(cl_idx),np.arange(N),s=2,c='#467eae',rasterized=True)
bx.set_ylim(N-.5,-.5);bx.set_xticks(range(len(classes)),classes,rotation=90,fontsize=9);bx.set_yticks([]);bx.set_title('같은 행의 SUBCLASS',fontsize=11)
cx.plot(burden.loc[order].values,np.arange(N),color='#467eae',lw=1)
cx.set_ylim(N-.5,-.5);cx.set_xlim(left=0);cx.set_yticks([]);cx.set_xlabel('전체 변이 유전자 셀 수');cx.set_title('같은 행의 전체 변이 수',fontsize=11)
fig.suptitle('E  전체 환자의 변이 유전자 수와 기능 단계 배치',fontsize=18,y=1.01)
fig.text(.02,-.02,SCOPE+' · 모든 환자 포함, 화면 해상도상 행 압축. 정확한 행별 정보는 E_all_patient_patterns.csv.',fontsize=11)
fig.tight_layout();save(fig,'E_all_patient_patterns')
# Group by exact count and stage signature; avoid an all-pairs O(N^2) scan.
pairs=[];selected=[]
for value,grp in patient_table.groupby('mutated_gene_cells',sort=True):
    reps=grp.assign(signature=grp[stages].fillna(-1).astype(int).astype(str).agg('|'.join,axis=1)).drop_duplicates('signature')
    if len(reps)<2:continue
    found=None
    for i,j in itertools.combinations(reps.index,2):
        dif=[s for s in stages if pd.notna(state.loc[i,s]) and pd.notna(state.loc[j,s]) and state.loc[i,s]!=state.loc[j,s]]
        if dif:found=(i,j,dif);break
    if found:
        i,j,dif=found;pairs.append(dict(patient_a=train.loc[i,'ID'],class_a=train.loc[i,'SUBCLASS'],patient_b=train.loc[j,'ID'],class_b=train.loc[j,'SUBCLASS'],equal_gene_cells=int(value),different_stages=','.join(dif)))
        if len(selected)<16:selected.extend([i,j])
pd.DataFrame(pairs).to_csv(OUT/'E_exact_count_pair_examples.csv',index=False)
if selected:
    z=state.loc[selected];fig,ax=plt.subplots(figsize=(12,max(5,len(selected)*.36)))
    texts=[['?' if pd.isna(v) else ('●' if v==1 else '0') for v in r] for r in z.values]
    labels=[f'{train.loc[i,"ID"]} | {train.loc[i,"SUBCLASS"]} | {burden[i]}개' for i in selected]
    heat(ax,z,stages,labels,texts)
    ax.set_title('E2  전체 변이 유전자 수가 같은 환자의 단계 차이 | 예시',pad=16)
    fig.text(.02,-.025,SCOPE+f' · 상세 예시 {len(selected)}명; 매 두 행이 같은 수의 비교쌍. 인과/예측력 결론 아님.',fontsize=11)
    fig.tight_layout();save(fig,'E2_exact_count_examples')
    display(pd.DataFrame(pairs).head(8))
else:display(Markdown('집계 불충분: 같은 전체 변이 유전자 수에서 관측 단계가 다른 비교쌍 없음.'))
display(Markdown('**현재 보이는 것:** 전체 변이 수 순서에서의 단계 분포와 정확히 같은 수의 비교쌍을 확인합니다. **다음 확인:** SUBCLASS 내 비교 및 동의변이 분리 후 반복되는지.'))

## 전체 데이터 구조와 읽기 기록
각 CSV 전체를 읽은 행 수를 아래 기록합니다. 원본의 결측과 WT를 분리하며 test의 클래스 정답은 없습니다. 아래에 표시하는 결측·원본 예시는 일부이지만 집계는 전체입니다.

기존 베이스라인의 `LabelEncoder`는 정답 클래스 번호, `OrdinalEncoder`는 유전자별 원본 문자열 전체의 범주 번호를 만듭니다. 범주 사전은 본 문자열을 구분하지만 모델 입력에서 아미노산 위치·치환 관계를 직접 표현하지 않으며 미지 문자열은 `-1`로 합쳐집니다. 이 숫자로 생물학적 크기·순서를 해석하거나 전체 피처 상관 히트맵을 만들지 않습니다.

In [ ]:
audit=[];missing_examples=[]
for name,df in frames.items():
    cols=[g for g in df.columns if g not in ['ID','SUBCLASS']]
    missing_n=0
    for gene in cols:
        mask=df[gene].str.strip().str.upper().isin(missing_markers);missing_n+=int(mask.sum())
        if len(missing_examples)<12:
            for i,value in df.loc[mask,gene].head(12-len(missing_examples)).items():
                missing_examples.append({'file':name,'ID':df.loc[i,'ID'],'gene':gene,'raw_repr':repr(value)})
    audit.append({'file':str(DATA/name),'rows_read':len(df),'columns':len(df.columns),'input_features':len(cols),'missing_gene_cells':missing_n if cols else None})
display(pd.DataFrame(audit));display(pd.DataFrame(missing_examples))
assert len(submission)==len(test) and submission.ID.tolist()==test.ID.tolist()
display(Markdown('train/test 유전자 이름·순서 및 test/제출양식 전체 ID 일치 확인.'))
import html
for name,df in frames.items():
    display(HTML('<details><summary>'+html.escape(name)+f': 전체 {len(df.columns)}열 이름</summary><p>'+html.escape(', '.join(df.columns))+'</p></details>'))
report={'scope':'all rows of original CSV files, executed only in Kaggle','files':audit,'n_train':N,
        'train_classes':{k:int(v) for k,v in counts.items()},'feature_name_order_match':True,
        'unique_mapped_genes':int(mapping.gene.nunique()),'mapped_present_genes':int(mapping.loc[mapping.present_in_train,'gene'].nunique()),
        'stage_present_genes':stage_genes,'mutated_gene_cells':int(mut.sum().sum()),'notation_tokens':len(annotations_all),
        'mapped_variant_cells':int(mapped[['ID','gene']].drop_duplicates().shape[0]),
        'exact_count_different_pattern_example_count':len(pairs),'exact_count_pair_examples':pairs[:8],
        'b_cells':b_rows,'overlap':overlap,'runtime':{'python':platform.python_version(),'pandas':pd.__version__,'matplotlib':matplotlib.__version__}}
(OUT/'full_eda_audit.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
pd.DataFrame(b_rows).to_csv(OUT/'B_subclass_stage_rates.csv',index=False)
assert len(annotations_all[['ID','gene']].drop_duplicates())==int(mut.sum().sum())
assert len(patient_table)==N
assert all(r['positive']<=r['known']<=r['class_n'] for r in b_rows)
assert np.array_equal(co,co.T)
print('Full EDA checks passed.',flush=True)

## XGB 베이스라인 | 고정 내부 검증 + 전체 train 학습
**새 최초 베이스라인이며 이전 20행 EDA와 성능 비교하지 않습니다.** 전체 입력 유전자를 사용하고 원본 노트북의 100 trees / learning_rate=0.1 / max_depth=6 / seed=42를 유지합니다. 실행 효율을 위해 CPU `hist`, 4 threads를 명시합니다.

1. SUBCLASS 층화 80% 학습 / 20% 검증. 동일 입력 행은 동일 구간에 배치해 중복 행 누수를 막습니다. 중복이 있으면 그룹 층화 5-fold의 첫 fold를 사용합니다.
2. 인코더는 **학습 구간만 fit**, 검증에는 transform. 미지 범주=-1, 결측=NaN이며 WT와 구분합니다.
3. 고정 100회 학습이므로 검증 결과로 모델을 선택하거나 조기 종료하지 않습니다. Macro-F1이 주 지표, accuracy·balanced accuracy와 클래스별 precision/recall/F1을 함께 보고합니다.
4. 평가 후 새 인코더·새 모델을 전체 train에 fit하여 저장합니다. 내부 검증 점수는 이 최종 모델의 독립 테스트 성능이 아닙니다. test 예측·제출은 실행하지 않습니다.

이는 탐색용 단일 분할 결과입니다. 전체 데이터 EDA를 본 후 진행하므로 앞으로의 피처 선택에는 별도 검증 설계가 필요합니다.

In [ ]:
import gc,time,joblib
import sklearn,xgboost as xgb
from sklearn.model_selection import train_test_split,StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder,OrdinalEncoder
from sklearn.metrics import f1_score,accuracy_score,balanced_accuracy_score,classification_report,confusion_matrix
SEED=42
# Missing markers are normalized to NaN only for model input, not the preserved raw strings.
X=train[features].copy().astype(object)
for g in features:X.loc[missing[g],g]=np.nan
le=LabelEncoder();y=le.fit_transform(train.SUBCLASS)
groups=pd.util.hash_pandas_object(train[features],index=False).astype(str)
duplicate_rows=int(groups.duplicated().sum())
conflicting_groups=int(pd.DataFrame({'g':groups,'label':train.SUBCLASS}).groupby('g').label.nunique().gt(1).sum())
if duplicate_rows:
    splitter=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
    tr,va=next(splitter.split(np.zeros(N),y,groups));split_method='StratifiedGroupKFold first of 5; identical feature rows grouped'
else:
    tr,va=train_test_split(np.arange(N),test_size=.2,stratify=y,random_state=SEED);split_method='stratified 80/20'
assert not set(train.ID.iloc[tr])&set(train.ID.iloc[va])
assert not set(groups.iloc[tr])&set(groups.iloc[va])
assert set(y[tr])==set(range(len(le.classes_)))
split=train[['ID','SUBCLASS']].copy();split['split']='validation';split.loc[tr,'split']='train'
split.to_csv(OUT/'baseline_split.csv',index=False)
enc=OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1,encoded_missing_value=np.nan,dtype=np.float32)
t0=time.time();Xt=enc.fit_transform(X.iloc[tr]);Xv=enc.transform(X.iloc[va])
params=dict(n_estimators=100,learning_rate=.1,max_depth=6,random_state=SEED,eval_metric='mlogloss',tree_method='hist',n_jobs=4)
model=xgb.XGBClassifier(**params)
print('Validation training started',len(tr),'train',len(va),'validation',flush=True)
model.fit(Xt,y[tr],eval_set=[(Xt,y[tr]),(Xv,y[va])],verbose=20)
pred=model.predict(Xv)
metrics={'macro_f1':float(f1_score(y[va],pred,average='macro',zero_division=0)),
         'accuracy':float(accuracy_score(y[va],pred)),'balanced_accuracy':float(balanced_accuracy_score(y[va],pred))}
majority=int(np.bincount(y[tr]).argmax());dummy=np.full(len(va),majority)
dummy_metrics={'macro_f1':float(f1_score(y[va],dummy,average='macro',zero_division=0)),'accuracy':float(accuracy_score(y[va],dummy))}
class_report=classification_report(y[va],pred,labels=np.arange(len(le.classes_)),target_names=le.classes_,output_dict=True,zero_division=0)
per_class=pd.DataFrame(class_report).T.loc[le.classes_]
per_class.to_csv(OUT/'baseline_per_class_metrics.csv',encoding='utf-8-sig')
val_preds=train.iloc[va][['ID','SUBCLASS']].copy();val_preds['predicted_SUBCLASS']=le.inverse_transform(pred.astype(int));val_preds.to_csv(OUT/'validation_predictions.csv',index=False)
history=model.evals_result();cm=confusion_matrix(y[va],pred,labels=np.arange(len(le.classes_)))
den=cm.sum(axis=1,keepdims=True);cmrate=np.divide(cm,den,out=np.zeros_like(cm,dtype=float),where=den!=0)
fig,(ax,bx)=plt.subplots(1,2,figsize=(17,7),gridspec_kw={'width_ratios':[1,1.5]})
for key,label,col in [('validation_0','학습','#467eae'),('validation_1','검증','#bf8b3a')]:ax.plot(history[key]['mlogloss'],label=label,color=col)
ax.set_xlabel('Boosting round');ax.set_ylabel('Multiclass log loss');ax.legend();ax.set_title('고정 100회 학습 곡선')
im=bx.imshow(cmrate,vmin=0,vmax=1,cmap='Blues');bx.set_xticks(range(len(le.classes_)),le.classes_,rotation=90,fontsize=8);bx.set_yticks(range(len(le.classes_)),le.classes_,fontsize=8)
bx.set_xlabel('예측 SUBCLASS');bx.set_ylabel('실제 SUBCLASS');bx.set_title('검증 혼동 행렬: 실제 클래스별 비율')
fig.colorbar(im,ax=bx,fraction=.025,pad=.02,label='실제 클래스 내 비율')
fig.suptitle('XGB 베이스라인 내부 검증',fontsize=18,y=1.02)
fig.text(.02,-.02,f'전체 train {N:,}명에서 학습 {len(tr):,}명 / 검증 {len(va):,}명 · seed=42 · Macro-F1={metrics["macro_f1"]:.4f} · test 성능 아님',fontsize=11)
fig.tight_layout();save(fig,'F_baseline_validation')
fig,ax=plt.subplots(figsize=(12,max(7,len(le.classes_)*.3)))
pos=np.arange(len(le.classes_));ax.barh(pos-.18,per_class['f1-score'],height=.34,label='F1',color='#467eae');ax.barh(pos+.18,per_class['recall'],height=.34,label='Recall',color='#bf8b3a')
ax.set_yticks(pos,[f'{cl} (n={int(per_class.loc[cl,"support"])})' for cl in le.classes_]);ax.invert_yaxis();ax.set_xlim(0,1);ax.legend();ax.set_xlabel('점수 0~1')
ax.set_title('클래스별 내부 검증 F1 · Recall',pad=16)
fig.text(.02,-.02,f'전체 train의 고정 내부 검증 {len(va):,}명 · 분모=각 클래스 검증 환자 수 · 단일 분할',fontsize=11)
fig.tight_layout();save(fig,'G_baseline_class_metrics')
display(pd.DataFrame([metrics,dummy_metrics],index=['XGB validation','majority baseline']));display(per_class)
baseline_report={'scope':'single internal holdout, not external test','metrics':metrics,'majority_baseline':dummy_metrics,'split_method':split_method,'seed':SEED,'train_rows':len(tr),'validation_rows':len(va),'features':len(features),'duplicate_feature_rows':duplicate_rows,'conflicting_duplicate_groups':conflicting_groups,'validation_unknown_cell_fraction':float(np.mean(Xv==-1)),'params':params,'validation_seconds':time.time()-t0,'versions':{'xgboost':xgb.__version__,'sklearn':sklearn.__version__},'full_train_model_status':'not_started'}
(OUT/'baseline_metrics.json').write_text(json.dumps(baseline_report,ensure_ascii=False,indent=2))
model.save_model(str(OUT/'xgb_validation_model.json'))
# Refit on all train with exactly the same fixed hyperparameters; no tuning.
del Xt,Xv,model,enc;gc.collect()
full_enc=OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1,encoded_missing_value=np.nan,dtype=np.float32)
Xall=full_enc.fit_transform(X)
full_model=xgb.XGBClassifier(**params)
print('Full train model started',N,'rows',flush=True)
t1=time.time();full_model.fit(Xall,y,verbose=False)
full_model.save_model(str(OUT/'xgb_full_train_model.json'))
joblib.dump({'encoder':full_enc,'label_encoder':le,'features':features,'missing_markers':sorted(missing_markers)},OUT/'xgb_full_preprocessing.joblib',compress=3)
baseline_report.update(full_train_model_status='COMPLETE',full_train_rows=N,full_train_seconds=time.time()-t1)
(OUT/'baseline_metrics.json').write_text(json.dumps(baseline_report,ensure_ascii=False,indent=2))
print('Full training complete. No test predictions or submission.',flush=True)

**다음 검증 가설:** 전체 변이 유전자 수가 비슷한 환자에서도 기능 단계별 배치·유형은 SUBCLASS에 따라 달라지는가? 현재 전체 데이터의 기술적 패턴을 새 피처로 시험할 때는 고정 검증 분할을 유지하고 인코더·피처 선택을 학습 구간에만 맞춥니다. 단일 분할 개선만으로 채택하지 않고 추가 분할에서 Macro-F1과 클래스별 recall/F1의 재현을 확인합니다.